#  Kiểm thử các trường hợp biên & Xử lý lỗi (QC Edge Cases & Error Handling)

Notebook này kiểm tra tính bền vững (robustness) của package `qc` trước các dữ liệu bất thường và lỗi đầu vào:
1. DataFrame rỗng hoặc chỉ có 1 dòng.
2. Cột toàn bộ missing (100% NaN) hoặc không có missing (0%).
3. Biểu đồ khi bộ lọc `min_*` loại bỏ toàn bộ dữ liệu (hiển thị thông điệp `_no_data`).
4. Kiểm tra các ngoại lệ `DatasetError` khi truyền tham số sai.



In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytest

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from qc import (
    plot_missing,
    plot_duplicate,
    plot_outlier_rate,
    plot_entity_coverage,
)
from qc._common import DatasetError

%matplotlib inline
print(" Sẵn sàng kiểm thử Edge Cases!")



## 1. Trường hợp DataFrame rỗng (`len(df) == 0`)



In [ ]:
empty_df = pd.DataFrame({
    "id": pd.Series(dtype=int),
    "val": pd.Series(dtype=float),
    "date": pd.Series(dtype="datetime64[ns]"),
})

# plot_missing trên DataFrame rỗng
fig1, tab1 = plot_missing(empty_df)
display(tab1)

# plot_duplicate trên DataFrame rỗng
fig2, tab2 = plot_duplicate(empty_df)
display(tab2)

# plot_outlier_rate trên DataFrame rỗng
fig3, tab3 = plot_outlier_rate(empty_df, cols=["val"])
display(tab3)

# plot_entity_coverage trên DataFrame rỗng
fig4, tab4 = plot_entity_coverage(empty_df, entity="id", time="date")
display(tab4)



## 2. Trường hợp bộ lọc `min_*` loại bỏ hết tất cả các cột
Khi không có cột nào thỏa điều kiện `min_missing` hoặc `min_rate`, biểu đồ phải hiển thị thông điệp văn bản ở giữa trục tọa độ mà không bị crash.



In [ ]:
# Tạo df chỉ có missing thấp (< 10%)
clean_df = pd.DataFrame({"col_a": [1, 2, 3, None], "col_b": [4, 5, 6, 7]})

# Đặt min_missing=0.50 (không cột nào đạt)
fig, table = plot_missing(clean_df, min_missing=0.50)
display(table)
print("Bảng rỗng:", table.empty)



## 3. Kiểm tra ngoại lệ Input Validation (`DatasetError`)

Package `qc` được thiết kế chặt chẽ để phát hiện và báo lỗi rõ ràng khi người dùng truyền tham số sai:



In [ ]:
def expect_dataset_error(func, *args, **kwargs):
    try:
        func(*args, **kwargs)
        raise AssertionError("Lẽ ra phải bắn DatasetError!")
    except DatasetError as e:
        print(f" Bắt được lỗi đúng kỳ vọng: {e}")

# 1. Truyền không phải DataFrame
expect_dataset_error(plot_missing, [1, 2, 3])

# 2. DataFrame có cột trùng tên
df_dup_cols = pd.DataFrame([[1, 2]], columns=["A", "A"])
expect_dataset_error(plot_missing, df_dup_cols)

# 3. Tham số rate ngoài [0, 1] hoặc sai kiểu
sample_df = pd.DataFrame({"x": [1, 2, 3]})
expect_dataset_error(plot_missing, sample_df, threshold=1.5)
expect_dataset_error(plot_missing, sample_df, min_missing=-0.1)
expect_dataset_error(plot_missing, sample_df, threshold="high")

# 4. Chọn cột không tồn tại
expect_dataset_error(plot_missing, sample_df, cols=["non_existent"])

# 5. Method outlier không hợp lệ
expect_dataset_error(plot_outlier_rate, sample_df, method="invalid_method")

# 6. Tham số k <= 0
expect_dataset_error(plot_outlier_rate, sample_df, k=-1)

# 7. Entity coverage: cột time không phải datetime
expect_dataset_error(plot_entity_coverage, sample_df, entity="x", time="x")

# 8. Entity coverage: freq không hợp lệ
sample_time_df = pd.DataFrame({"id": ["a", "b"], "dt": pd.to_datetime(["2024-01-01", "2024-01-02"])})
expect_dataset_error(plot_entity_coverage, sample_time_df, entity="id", time="dt", freq="INVALID_FREQ")



## 4. Tóm tắt kết quả
- Tất cả các trường hợp biên (DataFrame rỗng, cột toàn thiếu, lọc rỗng) đều được xử lý êm thuận.
- Cơ chế kiểm tra đầu vào (input validation) hoạt động chuẩn xác và trả về thông báo lỗi thân thiện bằng tiếng Việt.

